In [19]:
from datetime import date,timedelta
import httpx

today = date.today() - timedelta(days=1)
data = {
    "query": f"publication-date={today.strftime('%Y%m%d')}",
    "fields": [
        "publication-date",
        "notice-title",
        "procedure-type",
        "contract-nature",
        # TODO fix country
        # "country",
        "tender-value",
        "tender-value-cur",
        "classification-cpv",
        "organisation-contact-point-tenderer",
        "document-url-lot"
    ],
    "page": 1,
    "limit": 10,
    "scope": "ACTIVE",
    "checkQuerySyntax": False,
    "paginationMode": "ITERATION"
}

r = httpx.post('https://api.ted.europa.eu/v3/notices/search', json=data)
r.json()

{'notices': [{'organisation-contact-point-tenderer': ['Jiri Stanek',
    'Johan Håkansson'],
   'procedure-type': 'open',
   'classification-cpv': ['77310000',
    '77311000',
    '77314000',
    '77310000',
    '77311000',
    '77314000',
    '77310000',
    '77311000',
    '77314000',
    '77310000',
    '77311000',
    '77314000',
    '77310000',
    '77311000',
    '77314000',
    '77310000',
    '77311000',
    '77314000',
    '77300000',
    '77300000',
    '77300000',
    '77300000',
    '77300000',
    '77300000'],
   'publication-number': '32737-2025',
   'contract-nature': ['services',
    'services',
    'services',
    'services',
    'services',
    'services'],
   'publication-date': '2025-01-17+01:00',
   'links': {'xml': {'MUL': 'https://ted.europa.eu/en/notice/32737-2025/xml'},
    'pdf': {'BUL': 'https://ted.europa.eu/bg/notice/32737-2025/pdf',
     'SPA': 'https://ted.europa.eu/es/notice/32737-2025/pdf',
     'CES': 'https://ted.europa.eu/cs/notice/32737-2025/pdf',
 

In [1]:
import os

import httpx
from dotenv import load_dotenv
from oauthlib.oauth2 import BackendApplicationClient
from requests_oauthlib import OAuth2Session

load_dotenv(".env")


client_id = os.environ["PUBPROC_CLIENT_ID"]
client_secret = os.environ["PUBPROC_CLIENT_SECRET"]

url = "https://public.pr.fedservices.be/api/oauth2/token"

client = BackendApplicationClient(client_id=client_id)
oauth = OAuth2Session(client=client)

token = oauth.fetch_token(
    token_url=url,
    client_id=client_id,
    client_secret=client_secret,
    include_client_id=True,
)

print(token["access_token"], token["expires_at"])

/Users/mehmet/projects/proclogic_api/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


5O87eSXI5kbMXdQNdgiJw9CsEjJjMKQD0vMpuFvCWBiGQGppSBCP4N 1741571634.797709


In [ ]:
from datetime import date, timedelta

def get_nearest_business_day(date_obj: date = None) -> date:
    if date_obj is None:
        date_obj = date.today()

    if date_obj.weekday() == 5:  # Saturday
        return date_obj - timedelta(days=1)
    elif date_obj.weekday() == 6:  # Sunday
        return date_obj - timedelta(days=2)
    return date_obj

latest_business_day = get_nearest_business_day()

data = {
    "dispatch-date-from": "2025-01-01",
    "page": 1,
    "pageSize": 100,
}

headers = {
    "Authorization": f"Bearer {token['access_token']}",
    "BelGov-Trace-Id": "2ce83af9-d524-43a6-8d1c-b19dff051aed",
}
r = httpx.get('https://public.pr.fedservices.be/api/eProcurementSea/v1/search/publications', params=data, headers=headers)

publications = r.json()["publications"]
print("total count", r.json()["totalCount"])

if r.status_code == 200:
    totalCount = int(r.json()["totalCount"])
    pages = totalCount / 100 if totalCount % 100 == 0 else int(totalCount / 100) + 1

    if pages > 1:
        for i in range(2, pages + 1):
            data["page"] = i
            r = httpx.get(
                "https://public.pr.fedservices.be/api/eProcurementSea/v1/search/publications",
                params=data,
                headers=headers,
            )
            publications.extend(r.json()["publications"])
            break
        
publications

total count 6632
fetching page 2


[{'cpvAdditionalCodes': [{'code': '90721300-0',
    'descriptions': [{'language': 'NL',
      'text': 'Diensten voor bescherming tegen voedsel- of diervoederbesmetting'},
     {'language': 'DE',
      'text': 'Schutz vor Nahrungs- oder Futtermittelkontaminierung'},
     {'language': 'EN',
      'text': 'Food or feed contamination protection services'},
     {'language': 'FR',
      'text': "Services de protection contre la contamination de l'alimentation humaine ou de l'alimentation animale"}]}],
  'cpvMainCode': {'code': '90721300-0',
   'descriptions': [{'language': 'NL',
     'text': 'Diensten voor bescherming tegen voedsel- of diervoederbesmetting'},
    {'language': 'DE',
     'text': 'Schutz vor Nahrungs- oder Futtermittelkontaminierung'},
    {'language': 'EN',
     'text': 'Food or feed contamination protection services'},
    {'language': 'FR',
     'text': "Services de protection contre la contamination de l'alimentation humaine ou de l'alimentation animale"}]},
  'dispatchDa